# Preprocessing — MU-Glioma-Post (target = **First Progression**)

**Final-year project**: Literature-Augmented RAG for glioma recurrence
prediction (BEEP-style pipeline; Mistral-7B-Instruct + LoRA downstream).

## Scope

1. Load the raw clinical sheet (203 patients × 74 columns) and the
   raw `segmentation_volumes.xlsx` (multi-region statistics from the
   four BraTS MRI sequences).
2. Build the supervised target `y = Progression` on the **full
   203-patient cohort** (152 yes / 51 no, ~75 / 25 class balance).
3. Perform a full column-level **leakage audit** with empirical
   evidence for every post-event / mixed-timing field.
4. Impose a uniform **temporal gate** at the per-patient landmark
   `T = TTP1` (for y=1) or `T = last-follow-up` (for y=0). Any
   feature or MRI timepoint with day ≥ T is stripped.
5. Freeze the **5 feature groups** for the experiments the supervisor
   approved.
6. Produce patient-disjoint **stratified 70 / 15 / 15 splits** and
   persist them as `Train.csv`, `Validation.csv`, `Test.csv`.

### Experiment lineup (target = 1st progression)

| Exp | Feature set                                                        |
|-----|--------------------------------------------------------------------|
| 1   | Metadata only (demographics + diagnosis)                           |
| 2   | Metadata + Molecular markers                                       |
| 3   | Metadata + Treatment (**leaky salvage therapies removed**)         |
| 4   | Metadata + Molecular + Treatment                                   |

Note on Exp 3:

- **Exp 3 (Treatment)**: we keep only the initial post-surgery chemo
  and radiation (plus the first-surgery day). All *adjuvant / salvage*
  therapies (Immunotherapy, Brachytherapy, Additional Therapy, Other
  Therapy) are excluded entirely because they are predominantly
  started **after** the first-progression event — see §4.3.

### Why no Exp 5 (MRI / radiomics / follow-up) on this target

Two independent sources of leakage make a clean Exp 5 impossible on
the MU-Glioma-Post dataset when the target is 1st-progression:

1. **MRI timepoints**: 37 % of recorded Timepoint_1 scans were taken
   *after* the patient's first progression, and there is no scan-date
   filtering available in `segmentation_volumes.xlsx` (no timepoint
   tags) — see §5 and §7.
2. **Follow-up-schedule features**: using a per-patient landmark
   `T = TTP1 / censor` encodes the event time directly into the
   feature window, and any fixed landmark `X` (30 d, 90 d, 180 d)
   drops enough y=0 patients to worsen class balance (e.g. X=90 →
   142 / 203 kept, pos-rate 83 %).

The experimental breadth is instead recovered by two additional
ablation axes that are orthogonal to the feature-group axis:

- **Retrievers**: BEEP biencoder (faithful) vs `nomic-ai/nomic-embed-text-v1.5`
- **Rerankers**: PubMedBERT cross-encoder (BEEP — primary) vs MiniLM,
  MedCPT, ColBERT, BGE-M3 (ablations)

This lineup is fully leakage-free and maps directly onto the BEEP
pipeline for publication.


## 1. Imports and paths


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

SEED = 42

DATASET   = Path.cwd()
RAW_DIR   = DATASET / "Raw"
OUT_DIR   = DATASET / "Processed"
SPLIT_DIR = DATASET / "splits"
OUT_DIR.mkdir(exist_ok=True, parents=True)
SPLIT_DIR.mkdir(exist_ok=True, parents=True)

CLINICAL_XLSX = RAW_DIR / "MU-Glioma-Post_ClinicalData-July2025.xlsx"
SEGVOL_XLSX   = RAW_DIR / "segmentation_volumes.xlsx"
MRI_ROOT      = RAW_DIR / "MRI"

print(f"Clinical xlsx exists: {CLINICAL_XLSX.exists()}")
print(f"Segvol xlsx exists:   {SEGVOL_XLSX.exists()}")
print(f"MRI root exists:      {MRI_ROOT.exists()}  "
      f"({sum(1 for _ in MRI_ROOT.iterdir()) if MRI_ROOT.exists() else 0} patient dirs)")


## 2. Load raw clinical + segmentation-volume data


In [ ]:
clinical = pd.read_excel(CLINICAL_XLSX, sheet_name="MU Glioma Post")
print(f"Clinical sheet shape: {clinical.shape}")
print(f"Unique patients:      {clinical['Patient_ID'].nunique()}")
clinical.head(3)


In [ ]:
segvol = pd.read_excel(SEGVOL_XLSX)
print(f"segmentation_volumes shape: {segvol.shape}")
print("columns:", list(segvol.columns))
segvol.head(3)


## 3. Target construction

Target = binary `Progression` on the **full 203-patient cohort**
(every patient in MU-Glioma-Post has a `Progression` value, so the
cohort doesn't need filtering).


In [ ]:
clinical["Progression"] = pd.to_numeric(clinical["Progression"], errors="coerce")
clinical = clinical.dropna(subset=["Progression"]).reset_index(drop=True)
clinical["y"] = clinical["Progression"].astype(int)

print(f"Full cohort: N={len(clinical)}")
print(f"  y = 1 (progressed):      {int((clinical['y']==1).sum())}")
print(f"  y = 0 (no progression):  {int((clinical['y']==0).sum())}")
print(f"  Positive rate:           {clinical['y'].mean():.1%}")


## 4. Comprehensive leakage audit

### Prediction timepoint (landmark)

For every patient we set a per-patient landmark `T`:

- If `y = 1` (progressed):  `T = TTP1` (date of first progression)
- If `y = 0`:               `T = max(death, last_MRI)` — latest
                             observation we have for the patient

Any feature that can only be known **at or after `T`** is post-event
data and must be excluded, even if its marginal correlation with `y`
looks innocuous (see Hospice deep-dive in §4.2).

### Tier classification

| Tier | Meaning |
|------|---------|
| **T1** | Direct label / event-definition leak — MUST exclude |
| **T2** | Post-event or unknown at landmark — MUST exclude |
| **T3** | Mixed timing — keep column, **mask rows with start ≥ T** |
| **T4** | Safe baseline — include |


In [ ]:
LEAKAGE = [
    # ---------------- T4 safe baseline ----------------
    ("Patient_ID",                                           "T4", "Identifier"),
    ("Sex at Birth",                                         "T4", "Demographics (immutable)"),
    ("Race",                                                 "T4", "Demographics (immutable)"),
    ("Age at diagnosis",                                     "T4", "Baseline at t=0"),
    ("Primary Diagnosis",                                    "T4", "Baseline diagnosis"),
    ("Grade of Primary Brain Tumor",                         "T4", "Baseline diagnosis"),
    ("Stereotactic Biopsy before Surgical Resection",        "T4", "Pre-treatment procedure"),
    *[(c, "T4", "Baseline molecular marker") for c in [
        "IDH1 mutation","IDH2 mutation","1p/19q","ATRX mutation","MGMT methylation",
        "BRAF V600E mutation","TERT promoter mutation",
        "Chromosome 7 gain and Chromosome 10 loss","H3-3A mutation",
        "EGFR amplification","PTEN mutation","CDKN2A/B deletion","TP53 alteration",
        "Other mutations/alterations",
    ]],
    ("Previous Brain Tumor",                                 "T4", "Baseline history"),
    ("Type of previous brain tumor",                         "T4", "Baseline history"),
    ("Year of previous surgery",                             "T4", "Baseline history"),
    ("Grade of Previous brain tumor",                        "T4", "Baseline history"),
    ("Number of days from Diagnosis to First surgery or procedure ",
     "T4", "Initial surgery day"),
    *[(c, "T4", "MRI scan day — later gated by T (§5)") for c in [
        "Number of Days from Diagnosis to 1st MRI (Timepoint_1) ",
        "Number of Days from Diagnosis to 2nd MRI (Timepoint_2) ",
        "Number of Days from Diagnosis to 3rd MRI (Timepoint_3) ",
        "Number of Days from Diagnosis to 4th MRI (Timepoint_4) ",
        "Number of Days from Diagnosis to 5th MRI (Timepoint_5) ",
        "Number of Days from Diagnosis to 6th MRI (Timepoint_6) ",
    ]],

    # ---------------- T3 mixed timing (mask rows with start >= TTP1) ----------------
    ("Initial Chemo Therapy",                                "T3", "97.5 % pre-TTP1; mask 3 post-TTP1 rows"),
    ("Name of Initial Chemo Therapy",                        "T3", "Masked alongside chemo start day"),
    (" Number of days from Diagnosis to Initial Chemo Therapy Start date",
     "T3", "2.5 % (3/118) starts are post-TTP1"),
    (" Number of days from Diagnosis to Initial Chemo Therapy end date",
     "T3", "Masked alongside chemo start day"),
    ("Radiation Therapy",                                    "T3", "94.5 % pre-TTP1; mask 7 post-TTP1 rows"),
    ("Number of days from Diagnosis to Radiation Therapy Start date",
     "T3", "5.5 % (7/127) starts are post-TTP1"),
    ("Number of days from Diagnosis to Radiation Therapy end date",
     "T3", "Masked alongside RT start day"),
    ("Dose",                                                 "T3", "Masked alongside RT start day"),
    ("Number of Fractions",                                  "T3", "Masked alongside RT start day"),

    # ---------------- T2 post-event / unknown at landmark ----------------
    ("Hospice",                                              "T2",
     "End-of-life decision, recorded AFTER landmark. "
     "χ²=0.20 (p=0.65 vs 2nd-prog) but unknowable at T → EXCLUDE"),
    ("Overall Survival (Death)",                             "T2", "Post-event outcome"),
    ("Number of days from Diagnosis to death (Days)",        "T2", "Post-event date"),
    ("Multiple surgeries",                                   "T2",
     "Salvage surgery flag; no start day to verify it predates TTP1 → worst-case exclude"),
    # All salvage/adjuvant therapies — per supervisor: remove for 1st-prog target
    ("Additional Therapy",                                   "T2", "18.6 % started post-TTP1; per supervisor"),
    ("Cycle length of Additional Therapy (q days)",          "T2", "Companion of Additional Therapy"),
    ("Number of Days from Diagnosis to Starting Additional Therapy ",
     "T2", "Salvage start day for 1st-prog target"),
    ("Number of Days from Diagnosis to Complete Additional Therapy ",
     "T2", "Companion of Additional Therapy"),
    ("Number of Cycles of Additional Therapy",               "T2", "Companion of Additional Therapy"),
    ("Immuno therapy",                                       "T2",
     "82.0 % (41/50) started POST-TTP1 — predominantly salvage"),
    ("Cycle length of Immunotherapy (q days)",               "T2", "Companion of Immuno therapy"),
    ("Number of Days from Diagnosis to Start Immunotherapy ","T2", "82 % post-TTP1"),
    ("Number of Days from Diagnosis to Complete Immunotherapy ",
     "T2", "Companion of Immuno therapy"),
    ("Number of Cycles of Immunotherapy",                    "T2", "Companion of Immuno therapy"),
    ("Brachy therapy",                                       "T2",
     "66.7 % (12/18) inserted POST-TTP1 — predominantly salvage"),
    ("Number of Days from Diagnosis to the day of Insertion of Brachytherapy ",
     "T2", "67 % post-TTP1"),
    ("Other Types of Therapy (LITT, more chemo, proton therapy)",
     "T2", "30.5 % started post-TTP1"),
    ("Number of Days from Diagnosis to Start Other Additional Therapy ",
     "T2", "Companion of Other Therapy"),
    ("Number of Days from Diagnosis to Complete Other Additional Therapy ",
     "T2", "Companion of Other Therapy"),

    # ---------------- T1 direct label / event-definition leak ----------------
    ("Progression",                                          "T1", "Target label itself"),
    ("Time to First Progression (Days)",                     "T1", "Duration of the event we predict"),
    ("Type of 1st Progression",                              "T1", "Only filled when event occurred"),
    ("Number of days from Diagnosis to date of First Progression",
     "T1", "Date of the event we predict"),
    ("Second Progression/Recurrence",                        "T1", "Downstream of 1st-progression event"),
    ("Type of 2nd Progression",                              "T1", "Downstream of 1st-progression event"),
    ("Number of days from Diagnosis to date of Further Progression",
     "T1", "Downstream of 1st-progression event"),
    ("Treatment started after 2nd progression",              "T1", "Downstream of 1st-progression event"),
    ("Days from Diagnosis to new treatment",                 "T1", "Downstream of 1st-progression event"),
    ("2nd_Additional Therapy",                               "T1", "Downstream of 1st-progression event"),
    ("Cycle length of 2nd_Additional Therapy (q days)",      "T1", "Downstream of 1st-progression event"),
    ("Number of Days from Diagnosis to Starting 2nd_Additional Therapy ",
     "T1", "Downstream of 1st-progression event"),
    ("Number of Days from Dagnosis to Complete 2nd_Additional Therapy ",
     "T1", "Downstream of 1st-progression event"),
    ("Number of Cycles of 2nd_Additional Therapy",           "T1", "Downstream of 1st-progression event"),
]

leak_df = pd.DataFrame(LEAKAGE, columns=["column", "tier", "rationale"])
missing = set(clinical.columns) - set(leak_df["column"]) - {"y"}
extra   = set(leak_df["column"]) - set(clinical.columns)
assert not missing, f"Un-classified columns: {missing}"
assert not extra,   f"Manifest refers to non-existent columns: {extra}"
print(f"Covered {len(leak_df)} / {len(clinical.columns)-1} non-`y` columns (no orphans)")
print("\nTier counts:")
print(leak_df["tier"].value_counts().sort_index())


### 4.2 Hospice deep-dive (supervisor flagged)

The supervisor explicitly asked whether `Hospice` is leaky. The
empirical answer has two parts:

1. **Marginal correlation with `y`** — tested below.
2. **Temporal availability at landmark T** — **unknowable**.
   Hospice enrollment is a terminal-care decision made late in the
   disease course. For a y=1 patient it is effectively always set
   *after* T=TTP1; for a y=0 patient it would only be known well
   beyond our last observation.

Even when the marginal signal is weak, including it would encode the
labelling-pipeline's informative-censoring behaviour into the model
input — a classic reviewer red flag. **Decision: exclude (T2).**


In [ ]:
hospice_raw = pd.to_numeric(clinical["Hospice"], errors="coerce")
ct = pd.crosstab(hospice_raw.astype("Int64").astype(str),
                 clinical["y"], margins=True, dropna=False)
ct.columns = ["y=0 (no 1st prog)", "y=1 (1st prog)", "All"]
print("Hospice × y contingency table\n")
print(ct)

from scipy.stats import chi2_contingency
obs = pd.crosstab(hospice_raw.dropna().astype(int), clinical.loc[hospice_raw.notna(), "y"])
chi2, p, dof, _ = chi2_contingency(obs)
print(f"\nχ² test (filtering NaN): chi2={chi2:.3f}  p-value={p:.3f}  (dof={dof})")
print("\nInterpretation: " + (
    "weak/no marginal signal, but still excluded on temporal grounds."
    if p > 0.05 else
    "correlation present; still excluded because Hospice is set post-landmark."
))


### 4.3 Salvage / adjuvant therapy timing vs TTP1

For every therapy we compute the fraction of start dates that fall
**at or after** the per-patient TTP1. This is what drove the tiering
decisions in §4.1: anything with a non-trivial post-TTP1 rate is
either removed entirely (supervisor's instruction for the 1st-prog
target) or kept with row-level masking.


In [ ]:
cohort1 = clinical[clinical["y"] == 1].copy()
ttp1_1 = pd.to_numeric(cohort1["Number of days from Diagnosis to date of First Progression"],
                       errors="coerce")
therapy_pairs = [
    ("Initial Chemo",   " Number of days from Diagnosis to Initial Chemo Therapy Start date"),
    ("Radiation",       "Number of days from Diagnosis to Radiation Therapy Start date"),
    ("Additional",      "Number of Days from Diagnosis to Starting Additional Therapy "),
    ("Immunotherapy",   "Number of Days from Diagnosis to Start Immunotherapy "),
    ("Brachytherapy",   "Number of Days from Diagnosis to the day of Insertion of Brachytherapy "),
    ("Other",           "Number of Days from Diagnosis to Start Other Additional Therapy "),
]
rows = []
for name, col in therapy_pairs:
    d = pd.to_numeric(cohort1[col], errors="coerce")
    both = d.notna() & ttp1_1.notna()
    n_total = int(both.sum())
    n_pre   = int(((d <  ttp1_1) & both).sum())
    n_post  = int(((d >= ttp1_1) & both).sum())
    rows.append({
        "therapy": name, "n_with_start": n_total,
        "pre_TTP1": n_pre, "post_TTP1": n_post,
        "pct_post_TTP1": f"{(n_post/n_total*100) if n_total else 0:.1f}%",
        "tier_decision": {"Initial Chemo":"T3 (mask post)", "Radiation":"T3 (mask post)",
                          "Additional":"T2 (remove)", "Immunotherapy":"T2 (remove)",
                          "Brachytherapy":"T2 (remove)", "Other":"T2 (remove)"}[name],
    })
therapy_timing = pd.DataFrame(rows)
display(therapy_timing)


## 5. MRI-timepoint eligibility (strict temporal gate)

Per-patient landmark T:

- `y = 1` → `T = TTP1`
- `y = 0` → `T = max(death, last_mri)` (fall back to whichever exists)

Any MRI timepoint with `day ≥ T` is *post-event* and is stripped.


In [ ]:
MRI_DAY_COLS = [
    "Number of Days from Diagnosis to 1st MRI (Timepoint_1) ",
    "Number of Days from Diagnosis to 2nd MRI (Timepoint_2) ",
    "Number of Days from Diagnosis to 3rd MRI (Timepoint_3) ",
    "Number of Days from Diagnosis to 4th MRI (Timepoint_4) ",
    "Number of Days from Diagnosis to 5th MRI (Timepoint_5) ",
    "Number of Days from Diagnosis to 6th MRI (Timepoint_6) ",
]
mri = clinical[MRI_DAY_COLS].apply(pd.to_numeric, errors="coerce")
mri.columns = [f"TP{i}" for i in range(1, 7)]

ttp1   = pd.to_numeric(clinical["Number of days from Diagnosis to date of First Progression"],
                       errors="coerce")
death  = pd.to_numeric(clinical["Number of days from Diagnosis to death (Days)"], errors="coerce")
last_mri = mri.max(axis=1)

landmark = np.where(
    clinical["y"] == 1,
    ttp1,
    np.where(death.notna() & last_mri.notna(), np.maximum(death, last_mri),
             np.where(death.notna(), death, last_mri)),
)
landmark = pd.Series(landmark, index=clinical.index, name="landmark_T")

eligible_mask = mri.sub(landmark, axis=0) < 0
eligible_mask &= mri.notna() & (mri >= 0)
mri_clean = mri.where(eligible_mask, other=np.nan)

elig_count = eligible_mask.sum(axis=1)
print("=== MRI-timepoint eligibility (per patient count) ===")
print(elig_count.value_counts().sort_index().to_dict())
print(f"\nPatients with ≥1 eligible MRI: {(elig_count>=1).sum()} / {len(clinical)}")
print(f"Patients with 0 eligible MRI:  {(elig_count==0).sum()}")
print(f"\nTotal timepoints stripped (day ≥ T): "
      f"{int(((~eligible_mask) & mri.notna()).sum().sum())}")


## 6. Data-quality flags

Known dataset quirks discovered during scoping. The eligibility gate
from §5 already nullifies the offending *cells*, so we don't drop
patients — we just record flags for transparency.


In [ ]:
neg_any = (mri < 0).any(axis=1)
ooo_flag = pd.Series(False, index=clinical.index)
for idx, row in mri.iterrows():
    vals = [v for v in row.values if pd.notna(v)]
    if len(vals) >= 2 and any(vals[i+1] <= vals[i] for i in range(len(vals)-1)):
        ooo_flag.loc[idx] = True

post_death_flag = pd.Series(False, index=clinical.index)
for idx in clinical.index:
    d = death.loc[idx]
    if pd.notna(d) and any(pd.notna(v) and v > d for v in mri.loc[idx].values):
        post_death_flag.loc[idx] = True

dq = pd.DataFrame({
    "Patient_ID":      clinical["Patient_ID"],
    "y":               clinical["y"],
    "neg_mri_day":     neg_any.values,
    "out_of_order_tp": ooo_flag.values,
    "mri_after_death": post_death_flag.values,
})
dq["any_flag"] = dq[["neg_mri_day","out_of_order_tp","mri_after_death"]].any(axis=1)
print(f"Patients with any DQ flag: {int(dq['any_flag'].sum())} / {len(dq)}")
print(dq[dq["any_flag"]].head(25).to_string(index=False))
dq.to_csv(OUT_DIR / "data_quality_flags.csv", index=False)
print(f"\nSaved → {OUT_DIR/'data_quality_flags.csv'}")


## 7. Radiomics audit — **why we exclude it**

`segmentation_volumes.xlsx` has 334 rows × 13 columns:

| Columns |
|---------|
| `Patient ID`, `Label Id`, `Label Name`, `Number Of Voxels`, `Volume (mm^3)` |
| `Image mean/stdev (brain_t1c / brain_t1n / brain_t2f / brain_t2w)` |

Critically there is **no timepoint column**. Rows-per-patient
distribution is `{1:91, 2:46, 3:9, 4:16, 5:6, 6:5}` — a patient with
3 rows could be 3 scan-times, 3 regions, or a mix, and sample
inspection (`PatientID_0003` → 3 rows all `Label 1.0`) suggests rows
are **scans** (not regions), without a timepoint tag.

**Consequence**: we can't gate the radiomics by `scan_date < TTP1`.
For a 1st-progression target where 37 % of TP1 scans occur after the
event, this is a direct leakage risk. Radiomics is therefore
**excluded** for this pipeline. If a future pass re-extracts
radiomics directly from the raw NIfTI files (with timepoint tagging),
it can be added as a second Exp 5 variant.

We still report the patient-coverage so the exclusion is auditable.


In [ ]:
def _num_id(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.extract(r"(\d+)")[0], errors="coerce"
    ).astype("Int64")

segvol["pid_num"]    = _num_id(segvol["Patient ID"])
clinical["pid_num"]  = _num_id(clinical["Patient_ID"])
cohort_pids = set(clinical["pid_num"].dropna().astype(int))
seg_pids    = set(segvol["pid_num"].dropna().astype(int))
covered = cohort_pids & seg_pids
missing = cohort_pids - seg_pids
print(f"segmentation_volumes covers {len(covered)} / {len(cohort_pids)} cohort patients")
print(f"Missing ({len(missing)}): {sorted(missing)[:10]}{' ...' if len(missing)>10 else ''}")
print("\nPer-patient row count (= number of recorded scans, no timepoint labels):")
print(segvol.groupby('Patient ID').size().value_counts().sort_index())
print("\nWithout timepoint tags, post-TTP1 scan statistics cannot be filtered → radiomics EXCLUDED.")


## 8. Feature groups for the 4 experiments

Every group drops Tier T1 / T2 columns automatically. Tier T3 columns
enter only after row-level masking (§9). All four groups are
leakage-free at the diagnosis / initial-treatment landmark.


In [ ]:
DEMO = ["Sex at Birth", "Race", "Age at diagnosis"]
DIAG = ["Primary Diagnosis", "Grade of Primary Brain Tumor",
        "Stereotactic Biopsy before Surgical Resection",
        "Previous Brain Tumor", "Type of previous brain tumor",
        "Year of previous surgery", "Grade of Previous brain tumor"]
MOL  = ["IDH1 mutation","IDH2 mutation","1p/19q","ATRX mutation",
        "MGMT methylation","BRAF V600E mutation","TERT promoter mutation",
        "Chromosome 7 gain and Chromosome 10 loss","H3-3A mutation",
        "EGFR amplification","PTEN mutation","CDKN2A/B deletion",
        "TP53 alteration","Other mutations/alterations"]
TREAT = [
    "Number of days from Diagnosis to First surgery or procedure ",
    "Initial Chemo Therapy", "Name of Initial Chemo Therapy",
    " Number of days from Diagnosis to Initial Chemo Therapy Start date",
    " Number of days from Diagnosis to Initial Chemo Therapy end date",
    "Radiation Therapy",
    "Number of days from Diagnosis to Radiation Therapy Start date",
    "Number of days from Diagnosis to Radiation Therapy end date",
    "Dose", "Number of Fractions",
]

FEATURE_GROUPS = {
    "Exp1_metadata":                        DEMO + DIAG,
    "Exp2_metadata_molecular":              DEMO + DIAG + MOL,
    "Exp3_metadata_treatment":              DEMO + DIAG + TREAT,
    "Exp4_metadata_molecular_treatment":    DEMO + DIAG + MOL + TREAT,
}

for k, v in FEATURE_GROUPS.items():
    print(f"{k:<42}  {len(v):3d} cols")

with open(OUT_DIR / "feature_groups.json", "w") as f:
    json.dump(FEATURE_GROUPS, f, indent=2)
print(f"\nSaved → {OUT_DIR/'feature_groups.json'}")


## 9. Build the clean feature matrix

1. Drop every T1 / T2 column.
2. **Mask** T3 rows where any treatment's start day ≥ TTP1 (sets start
   day and all companion fields to NaN for that patient). For
   Progression=0 patients TTP1 is NaN so the mask never fires — their
   initial-treatment data are kept as recorded.
3. **Mask** treatment end-day cells that fall after TTP1, even when the
   treatment start was pre-TTP1.
4. **Mask** first-surgery day cells that fall after TTP1.
5. Save `clean_clinical.csv` (4-experiment safe feature space).


In [ ]:
t12_cols = leak_df.loc[leak_df["tier"].isin(["T1","T2"]), "column"].tolist()
clean = clinical.drop(columns=t12_cols).copy()
print(f"Dropped {len(t12_cols)} T1/T2 columns")
print(f"Remaining columns: {clean.shape[1]} (including `y`)")

T3_MASK_SETS = [
    (" Number of days from Diagnosis to Initial Chemo Therapy Start date",
     ["Initial Chemo Therapy", "Name of Initial Chemo Therapy",
      " Number of days from Diagnosis to Initial Chemo Therapy end date"]),
    ("Number of days from Diagnosis to Radiation Therapy Start date",
     ["Radiation Therapy",
      "Number of days from Diagnosis to Radiation Therapy end date",
      "Dose", "Number of Fractions"]),
]
ttp1_clean = pd.to_numeric(clinical["Number of days from Diagnosis to date of First Progression"],
                           errors="coerce")
masked_total = 0
end_day_masked_total = 0
for start_col, companions in T3_MASK_SETS:
    d = pd.to_numeric(clean[start_col], errors="coerce")
    mask = d.notna() & ttp1_clean.notna() & (d >= ttp1_clean)
    n = int(mask.sum())
    masked_total += n
    clean.loc[mask, start_col] = np.nan
    for c in companions:
        clean.loc[mask, c] = np.nan
    print(f"  Masked {n:3d} post-TTP1 rows in {start_col.strip()!r}")
print(f"\nTotal post-TTP1 rows masked: {masked_total}")

T3_END_DAY_COLS = [
    " Number of days from Diagnosis to Initial Chemo Therapy end date",
    "Number of days from Diagnosis to Radiation Therapy end date",
]
for col in T3_END_DAY_COLS:
    end_day = pd.to_numeric(clean[col], errors="coerce")
    mask = end_day.notna() & ttp1_clean.notna() & (end_day >= ttp1_clean)
    n = int(mask.sum())
    end_day_masked_total += n
    clean.loc[mask, col] = np.nan
    print(f"  Masked {n:3d} post-TTP1 end-day cells in {col.strip()!r}")
print(f"Total post-TTP1 end-day cells masked: {end_day_masked_total}")

surgery_col = "Number of days from Diagnosis to First surgery or procedure "
surgery_day = pd.to_numeric(clean[surgery_col], errors="coerce")
surgery_mask = surgery_day.notna() & ttp1_clean.notna() & (surgery_day > ttp1_clean)
clean.loc[surgery_mask, surgery_col] = np.nan
print(f"Masked {int(surgery_mask.sum())} post-TTP1 first-surgery day cells")

clean = clean.drop(columns=["pid_num"])
# Also drop the raw MRI-day columns — not in any experiment group, kept
# only in the raw sheet so downstream scripts don't accidentally pull them in.
mri_day_cols_in_clean = [c for c in clean.columns if "MRI (Timepoint_" in c]
clean = clean.drop(columns=mri_day_cols_in_clean)
print(f"Dropped {len(mri_day_cols_in_clean)} raw MRI-day columns from the feature matrix")
print(f"\nFinal clean matrix: {clean.shape}")
clean.head(3)


In [ ]:
clean.to_csv(OUT_DIR / "clean_clinical.csv", index=False)
print(f"Saved → {OUT_DIR/'clean_clinical.csv'}")


## 10. Train / Validation / Test splits

- **70 / 15 / 15**, stratified on `y`.
- Patient-disjoint (split on `Patient_ID`).
- Fixed seed 42 for reproducibility across the five experiments.
- With the ~75 / 25 class ratio we retain balanced pos-rates across
  the three splits (all close to 75 %).


In [ ]:
from sklearn.model_selection import train_test_split

ASSIGN_PATH = SPLIT_DIR / "split_assignments.csv"

def _load_existing_assignments():
    if ASSIGN_PATH.exists():
        assign_df = pd.read_csv(ASSIGN_PATH)
        return {s: assign_df.loc[assign_df["split"].eq(s), "Patient_ID"].tolist()
                for s in ("Train", "Validation", "Test")}
    if all((SPLIT_DIR / f"{s}.csv").exists() for s in ("Train", "Validation", "Test")):
        return {s: pd.read_csv(SPLIT_DIR / f"{s}.csv")["Patient_ID"].tolist()
                for s in ("Train", "Validation", "Test")}
    return None

existing_assign = _load_existing_assignments()
if existing_assign is not None:
    current = set(clean["Patient_ID"])
    assigned = set().union(*[set(v) for v in existing_assign.values()])
    assert current == assigned, (
        "Existing split assignments do not match the current First_Recur cohort. "
        "Refusing to silently resample patient IDs."
    )
    train_pids = existing_assign["Train"]
    valid_pids = existing_assign["Validation"]
    test_pids = existing_assign["Test"]
    print(f"Reusing frozen split assignments from {ASSIGN_PATH if ASSIGN_PATH.exists() else SPLIT_DIR}")
else:
    y = clean["y"].astype(int).to_numpy()
    pids = clean["Patient_ID"].to_numpy()

    train_pids, rest_pids, y_train, y_rest = train_test_split(
        pids, y, test_size=0.30, stratify=y, random_state=SEED
    )
    valid_pids, test_pids, _, _ = train_test_split(
        rest_pids, y_rest, test_size=0.50, stratify=y_rest, random_state=SEED
    )
    print("Generated new split assignments with fixed SEED=42")

def _summ(name, ids):
    sub = clean[clean["Patient_ID"].isin(ids)]
    print(f"  {name:<15} N={len(sub):3d}  y=1:{int((sub['y']==1).sum())}  "
          f"y=0:{int((sub['y']==0).sum())}  pos_rate={sub['y'].mean():.1%}")
_summ("Train",      train_pids)
_summ("Validation", valid_pids)
_summ("Test",       test_pids)

train = clean[clean["Patient_ID"].isin(train_pids)].reset_index(drop=True)
valid = clean[clean["Patient_ID"].isin(valid_pids)].reset_index(drop=True)
test  = clean[clean["Patient_ID"].isin(test_pids)].reset_index(drop=True)

assert set(train["Patient_ID"]).isdisjoint(valid["Patient_ID"])
assert set(train["Patient_ID"]).isdisjoint(test["Patient_ID"])
assert set(valid["Patient_ID"]).isdisjoint(test["Patient_ID"])
assert len(train) + len(valid) + len(test) == len(clean)

train.to_csv(SPLIT_DIR / "Train.csv",      index=False)
valid.to_csv(SPLIT_DIR / "Validation.csv", index=False)
test.to_csv(SPLIT_DIR / "Test.csv",       index=False)
pd.concat([
    pd.DataFrame({"Patient_ID": train["Patient_ID"], "split": "Train"}),
    pd.DataFrame({"Patient_ID": valid["Patient_ID"], "split": "Validation"}),
    pd.DataFrame({"Patient_ID": test["Patient_ID"],  "split": "Test"}),
], ignore_index=True).sort_values("Patient_ID").to_csv(ASSIGN_PATH, index=False)
print(f"\nSaved splits → {SPLIT_DIR}/  (Train.csv, Validation.csv, Test.csv)")
print(f"Saved frozen split assignments → {ASSIGN_PATH}")


## 11. Persist manifests

Everything downstream (literature retrieval, prompt builder, LoRA
trainer) reads these manifests so the leakage policy is enforced in
exactly one place.


In [ ]:
leak_df.to_csv(OUT_DIR / "leakage_manifest.csv", index=False)
print(f"Saved → {OUT_DIR/'leakage_manifest.csv'}  ({len(leak_df)} rows)")

elig_out = pd.DataFrame({
    "Patient_ID":      clinical["Patient_ID"],
    "y":               clinical["y"].values,
    "landmark_T":      landmark.values,
    **{f"TP{i}_day":      mri[f"TP{i}"].values                       for i in range(1,7)},
    **{f"TP{i}_eligible": eligible_mask[f"TP{i}"].astype(int).values for i in range(1,7)},
    "num_eligible_tp": eligible_mask.sum(axis=1).values,
})
elig_out.to_csv(OUT_DIR / "mri_eligibility.csv", index=False)
print(f"Saved → {OUT_DIR/'mri_eligibility.csv'}  ({len(elig_out)} rows)")

summary = {
    "target": "1st_progression (y = Progression)",
    "cohort_size": int(len(clinical)),
    "y_yes": int((clinical['y']==1).sum()),
    "y_no":  int((clinical['y']==0).sum()),
    "pos_rate": round(float(clinical['y'].mean()), 4),
    "columns_total": int(len(leak_df)),
    "columns_dropped_T1": int((leak_df['tier']=='T1').sum()),
    "columns_dropped_T2": int((leak_df['tier']=='T2').sum()),
    "columns_masked_T3": int((leak_df['tier']=='T3').sum()),
    "columns_safe_T4":   int((leak_df['tier']=='T4').sum()),
    "patients_mri_eligible_ge_1": int((eligible_mask.sum(axis=1)>=1).sum()),
    "patients_mri_eligible_eq_0": int((eligible_mask.sum(axis=1)==0).sum()),
    "radiomics_coverage": {"covered": len(covered), "missing": len(missing)},
    "data_quality_flags": {
        "neg_mri_day":     int(dq['neg_mri_day'].sum()),
        "out_of_order_tp": int(dq['out_of_order_tp'].sum()),
        "mri_after_death": int(dq['mri_after_death'].sum()),
    },
    "splits": {
        "train":      int(len(train)),
        "validation": int(len(valid)),
        "test":       int(len(test)),
        "seed":       SEED,
        "ratio":      "70/15/15 stratified on y",
    },
    "experiments": list(FEATURE_GROUPS.keys()),
    "experiment_count": len(FEATURE_GROUPS),
    "exp5_status": "DROPPED (MRI/radiomics leaky; follow-up-schedule features also leaky; see §7/§8).",
}
with open(OUT_DIR / "preprocessing_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved → {OUT_DIR/'preprocessing_summary.json'}")
print(json.dumps(summary, indent=2))


## 12. Outputs

| File | Purpose |
|------|---------|
| `Processed/clean_clinical.csv`          | Full cohort (203 × `k`), leakage-free, T3 masked, MRI-day-eligibility attached |
| `splits/Train.csv`                      | Training split (70 %) |
| `splits/Validation.csv`                 | Validation split (15 %) |
| `splits/Test.csv`                       | Test split (15 %) |
| `Processed/leakage_manifest.csv`        | Per-column tier + rationale (auditable trail) |
| `Processed/mri_eligibility.csv`         | Per-patient MRI-day eligibility table |
| `Processed/data_quality_flags.csv`      | Negative-day / post-death / OOO flags |
| `Processed/feature_groups.json`         | Feature lists for Exp 1-5 |
| `Processed/preprocessing_summary.json`  | Machine-readable run summary |

Next notebook: **`EDA.ipynb`** — visual analysis of the cohort,
leakage column distributions, treatment pathways, MRI timepoint audit
and radiomics per sequence / per region, all on the 1st-progression
target with the landmark gate applied.
